# Librerías y configuración inicial

In [1]:
import json
import pandas as pd
from datetime import datetime, date
from pathlib import Path

TZ = "Europe/Madrid"

# Carga de datos y transformación a datasets

In [2]:
def load_file(filename, extension = 'csv'):
    current = Path.cwd()
    path = current.parent / "data" / "raw_data" / extension

    file_path = path / f"{filename}.json" if extension == 'json' else path / f"{filename}.csv"

    if not file_path.exists():
        raise FileNotFoundError(f"[ERROR] No se encontró el archivo: {file_path}")

    if extension == 'json':
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
    else:
        data = pd.read_csv(file_path, sep=";")

    print(f"[INFO] Archivo cargado desde: {file_path}")

    return data

In [3]:
def parse_demanda(json_list):

    if isinstance(json_list, dict):
        json_list = [json_list]

    rows = []

    for json_data in json_list:

        last_update = datetime.fromisoformat(json_data["data"]["attributes"]["last-update"])
        values = json_data.get("included", [])[0].get("attributes", {}).get("values", [])

        for v in values:
            rows.append({
                "datetime"    : datetime.fromisoformat(v["datetime"]),
                "demanda"     : v["value"],
                "last_update" : last_update
            })
    
    return pd.DataFrame(rows).sort_values("datetime").reset_index(drop=True)

In [4]:
def parse_intercambios(json_list):

    if isinstance(json_list, dict):
        json_list = [json_list]

    rows = []

    for json_data in json_list:

        for country_block in json_data.get("included", []):
            
            pais = country_block.get("id")  # Francia, Portugal, Marruecos...
            
            for item in country_block.get("attributes", {}).get("content", []):

                # No guardamos los agregados
                if item.get("attributes", {}).get("composite") is True:
                    continue

                tipo   = item.get("type")  # Exportación, Importación, saldo
                values = item.get("attributes", {}).get("values", [])

                for v in values:
                    rows.append({
                        "datetime" : datetime.fromisoformat(v["datetime"]),
                        "pais"     : pais,
                        "tipo"     : tipo,
                        "valor"    : v["value"]
                    })

    return pd.DataFrame(rows).sort_values(["datetime", "pais", "tipo"]).reset_index(drop=True)

In [5]:
def parse_balance(json_list, subdata):

    if isinstance(json_list, dict):
        json_list = [json_list]

    rows = []

    for json_data in json_list:
        for bloque in json_data.get("included", []):

            # Solo queremos generación
            categoria = bloque.get("type")  # Renovable / No-Renovable / Almacenamiento / Demanda
            if subdata == 'generacion' and categoria not in ["Renovable", "No-Renovable"]:
                continue
            elif subdata == 'almacenamiento' and categoria != "Almacenamiento":
                continue

            for item in bloque.get("attributes", {}).get("content", []):

                # No guardamos los agregados
                if item.get("attributes", {}).get("composite") is True:
                    continue

                tecnologia = item.get("type")
                values     = item.get("attributes", {}).get("values", [])

                for v in values:
                    if subdata == 'generacion':
                        rows.append({
                            "datetime"     : datetime.fromisoformat(v["datetime"]),
                            "tipo_energia" : categoria,   # Renovable / No-Renovable
                            "tecnologia"   : tecnologia,
                            "valor"        : v["value"]
                        })
                    else:
                        rows.append({
                            "datetime"   : datetime.fromisoformat(v["datetime"]),
                            "tecnologia" : tecnologia,
                            "valor"      : v["value"]
                        })

    if subdata == 'generacion':
        return pd.DataFrame(rows).sort_values(["datetime", "tipo_energia", "tecnologia"]).reset_index(drop=True)
    else:
        return pd.DataFrame(rows).sort_values(["datetime", "tecnologia"]).reset_index(drop=True)


In [6]:
def parse_almacenamiento(json_list):

    if isinstance(json_list, dict):
        json_list = [json_list]

    rows = []

    for json_data in json_list:
        for bloque in json_data.get("included", []):

            if bloque.get("type") != "Almacenamiento":
                continue

            for item in bloque.get("attributes", {}).get("content", []):

                # No guardamos los agregados
                if item.get("attributes", {}).get("composite") is True:
                    continue

                tecnologia = item.get("type")
                values     = item.get("attributes", {}).get("values", [])

                for v in values:
                    rows.append({
                        "datetime"   : datetime.fromisoformat(v["datetime"]),
                        "tecnologia" : tecnologia,
                        "valor"      : v["value"]
                    })

    return pd.DataFrame(rows).sort_values(["datetime", "tecnologia"]).reset_index(drop=True)


In [7]:
json_balance      = load_file("balance", extension = 'json')
json_demanda      = load_file("demanda", extension = 'json')
json_intercambios = load_file("intercambios", extension = 'json')

[INFO] Archivo cargado desde: c:\UOC\TFM\data\raw_data\json\balance.json
[INFO] Archivo cargado desde: c:\UOC\TFM\data\raw_data\json\demanda.json
[INFO] Archivo cargado desde: c:\UOC\TFM\data\raw_data\json\intercambios.json


In [8]:
df_demanda        = parse_demanda(json_demanda)
df_generacion     = parse_balance(json_balance, subdata='generacion')
df_almacenamiento = parse_balance(json_balance, subdata='almacenamiento')
df_intercambio    = parse_intercambios(json_intercambios)

In [9]:
df_price = load_file("precio_marginal", extension = 'csv')
df_tech  = load_file("energia_casada", extension = 'csv')

[INFO] Archivo cargado desde: c:\UOC\TFM\data\raw_data\csv\precio_marginal.csv
[INFO] Archivo cargado desde: c:\UOC\TFM\data\raw_data\csv\energia_casada.csv


# Exploración inicial de los datos

In [10]:
dfs = {
    "Demanda"                   : df_demanda,
    "Precio marginal"           : df_price,
    "Almacenamiento"            : df_almacenamiento,
    "Intercambios"              : df_intercambio,
    "Energía generada por tech" : df_generacion,
    "Energía casada por tech"   : df_tech
}

In [11]:
print("Dimensiones de cada dataset:")
for name, df in dfs.items():
    print(f"{name:25s} : {df.shape}")

Dimensiones de cada dataset:
Demanda                   : (61368, 3)
Precio marginal           : (10044, 27)
Almacenamiento            : (7232, 3)
Intercambios              : (20456, 4)
Energía generada por tech : (28322, 4)
Energía casada por tech   : (53976, 14)


In [12]:
print("Columnas de cada dataset:")
for name, df in dfs.items():
    print(f"\n{name}:")
    print(df.columns.tolist())

Columnas de cada dataset:

Demanda:
['datetime', 'demanda', 'last_update']

Precio marginal:
['DATE', 'CONCEPT', 'H1', 'H2', 'H3', 'H4', 'H5', 'H6', 'H7', 'H8', 'H9', 'H10', 'H11', 'H12', 'H13', 'H14', 'H15', 'H16', 'H17', 'H18', 'H19', 'H20', 'H21', 'H22', 'H23', 'H24', 'H25']

Almacenamiento:
['datetime', 'tecnologia', 'valor']

Intercambios:
['datetime', 'pais', 'tipo', 'valor']

Energía generada por tech:
['datetime', 'tipo_energia', 'tecnologia', 'valor']

Energía casada por tech:
['DATE', 'HOUR', 'COAL', 'FUEL_GAS', 'SELF_PRODUCER', 'NUCLEAR', 'HYDRO', 'COMBINED_CYCLE', 'WIND', 'THERMAL_SOLAR', 'PHOTOVOLTAIC_SOLAR', 'RESIDUALS', 'IMPORT', 'IMPORT_WITHOUT_MIBEL']


In [13]:
print("Tipado de las columnas de cada dataset:")
for name, df in dfs.items():
    print(f"\n{name}:")
    print(df.dtypes)

Tipado de las columnas de cada dataset:

Demanda:
datetime        object
demanda        float64
last_update     object
dtype: object

Precio marginal:
DATE        object
CONCEPT     object
H1         float64
H2         float64
H3         float64
H4         float64
H5         float64
H6         float64
H7         float64
H8         float64
H9         float64
H10        float64
H11        float64
H12        float64
H13        float64
H14        float64
H15        float64
H16        float64
H17        float64
H18        float64
H19        float64
H20        float64
H21        float64
H22        float64
H23        float64
H24        float64
H25        float64
dtype: object

Almacenamiento:
datetime       object
tecnologia     object
valor         float64
dtype: object

Intercambios:
datetime     object
pais         object
tipo         object
valor       float64
dtype: object

Energía generada por tech:
datetime         object
tipo_energia     object
tecnologia       object
valor           

In [14]:
print("Número de filas duplicadas en cada dataset:")
for name, df in dfs.items():
    print(f"\n{name:26s}: {df.duplicated().sum()}")

Número de filas duplicadas en cada dataset:

Demanda                   : 0

Precio marginal           : 0

Almacenamiento            : 0

Intercambios              : 0

Energía generada por tech : 0

Energía casada por tech   : 0


In [15]:
print("Número de nulos presentes en cada dataset:")
for name, df in dfs.items():
    print(f"\n{name}:")
    display(df.isna().sum())

Número de nulos presentes en cada dataset:

Demanda:


datetime       0
demanda        0
last_update    0
dtype: int64


Precio marginal:


DATE          0
CONCEPT       0
H1            0
H2            0
H3            0
H4            0
H5            0
H6            0
H7            0
H8            0
H9            0
H10           0
H11           0
H12           0
H13           0
H14           0
H15           0
H16           0
H17           0
H18           0
H19           0
H20           0
H21           0
H22           0
H23           0
H24          28
H25        9836
dtype: int64


Almacenamiento:


...

datetime    0
pais        0
tipo        0
valor       0
dtype: int64


Energía generada por tech:


datetime        0
tipo_energia    0
tecnologia      0
valor           0
dtype: int64


Energía casada por tech:


DATE                        0
HOUR                        0
COAL                    29352
FUEL_GAS                53976
SELF_PRODUCER           53976
NUCLEAR                     0
HYDRO                       0
COMBINED_CYCLE          13003
WIND                        0
THERMAL_SOLAR            2552
PHOTOVOLTAIC_SOLAR          1
RESIDUALS                   0
IMPORT                  52044
IMPORT_WITHOUT_MIBEL        0
dtype: int64

In [16]:
print("Muestreo de cada dataset:")
for name, df in dfs.items():
    print(f"\n{name}:")
    display(df.sample(3))

Muestreo de cada dataset:

Demanda:


,datetime,demanda,last_update
58568,2025-09-06 09:00:00+02:00,25060.716,2026-01-13 22:17:41+01:00
16836,2020-12-02 12:00:00+01:00,33763.533,2021-11-10 11:21:34+01:00
11681,2020-05-01 18:00:00+02:00,20127.567,2021-04-12 10:54:32+02:00



Precio marginal:


,DATE,CONCEPT,H1,H2,H3,H4,H5,H6,H7,H8,...,H16,H17,H18,H19,H20,H21,H22,H23,H24,H25
5084,2022-06-25,PRICE_SP,185.00,177.61,169.56,164.16,160.00,155.43,148.74,142.58,...,77.10,65.71,66.47,77.10,92.99,119.73,154.56,160.38,145.0,NaN
73,2019-01-19,PRICE_PT,70.16,68.40,64.11,59.20,55.55,56.60,57.96,62.84,...,61.73,62.93,66.40,68.71,70.00,67.56,66.42,66.02,63.5,NaN
4618,2022-02-28,ENER_IB,20789.40,19144.70,18267.60,17840.60,17907.10,18631.70,21322.40,24781.00,...,28459.10,25966.90,25858.20,26354.20,28946.10,30218.30,29324.10,26522.60,23364.5,NaN



Almacenamiento:


...

,datetime,pais,tipo,valor
16314,2024-08-01 00:00:00+02:00,Francia,Exportación,-3127.129
12814,2023-05-21 00:00:00+02:00,Portugal,Exportación,-51045.778
6968,2021-05-21 00:00:00+02:00,Andorra,Exportación,0.000



Energía generada por tech:


,datetime,tipo_energia,tecnologia,valor
24978,2025-03-18 00:00:00+01:00,No-Renovable,Nuclear,171186.429
13374,2022-04-29 00:00:00+02:00,No-Renovable,Cogeneración,66020.210
25784,2025-05-30 00:00:00+02:00,Renovable,Hidráulica,99717.515



Energía casada por tech:


,DATE,HOUR,COAL,FUEL_GAS,SELF_PRODUCER,NUCLEAR,HYDRO,COMBINED_CYCLE,WIND,THERMAL_SOLAR,PHOTOVOLTAIC_SOLAR,RESIDUALS,IMPORT,IMPORT_WITHOUT_MIBEL
30962,14/07/2022,4,784.0,NaN,NaN,6897.7,542.4,12781.6,4685.8,333.0,0.1,2708.3,NaN,0.0
27588,23/02/2022,13,807.0,NaN,NaN,7086.3,714.5,5600.6,3579.2,854.2,8401.6,4656.3,NaN,3509.7
47333,26/05/2024,7,NaN,NaN,NaN,4034.1,4778.6,NaN,2696.6,533.0,127.9,3538.3,NaN,3191.0


# Preprocesado de los datos

## Datos en formato horario

In [17]:
df_demanda['datetime'] = pd.to_datetime(df_demanda['datetime'], utc=True)
df_demanda['datetime'] = df_demanda['datetime'].dt.tz_convert(TZ)

df_demanda = df_demanda[['datetime', 'demanda']].sort_values('datetime', ignore_index=True)

In [18]:
print(f"Posibles valores de tipo de dato:\n{df_price['CONCEPT'].unique()}")

Posibles valores de tipo de dato:
['PRICE_SP' 'PRICE_PT' 'ENER_IB' 'ENER_IB_BILLAT']


In [19]:
# Tenemos que hacer un "melt" sobre df_price, porque cada columna es una hora distinta (de formato ancho a largo)
price_long = df_price.melt(id_vars=['DATE', 'CONCEPT'], var_name='HORA', value_name='valor')

# Convertimos los valores obtenidos por las columnas (H1, H2...) a enteros y eliminamos la hora 25
price_long['HORA'] = price_long['HORA'].str.replace('H', '').astype(int)
price_long = price_long[price_long['HORA'] <= 24]

# Construimos el datetime y lo localizamos a la hora de la península
price_long['DATE']     = pd.to_datetime(price_long['DATE'])
price_long['datetime'] = price_long['DATE'] + pd.to_timedelta(price_long['HORA'] - 1, unit='h')
price_long['datetime'] = price_long['datetime'].dt.tz_localize(TZ, ambiguous="NaT", nonexistent="NaT")

# Dividimos en datasets distintos el precio marginal español y la energía casada del mercado ibérico (España y Portugal)
df_price_sp = (
    price_long[price_long['CONCEPT'] == 'PRICE_SP']
    .rename(columns={'valor': 'precio'})
    [['datetime', 'precio']]
    .sort_values('datetime', ignore_index=True)
)

df_ener_ib = (
    price_long[price_long['CONCEPT'] == 'ENER_IB']
    .rename(columns={'valor': 'energia_mercado'})
    [['datetime', 'energia_mercado']]
    .sort_values('datetime', ignore_index=True)
)

In [20]:
# Construimos el datetime y lo localizamos a la hora de la península
df_tech['DATE']     = pd.to_datetime(df_tech['DATE'], dayfirst=True)
df_tech['datetime'] = df_tech['DATE'] + pd.to_timedelta(df_tech['HOUR'] - 1, unit='h')
df_tech['datetime'] = df_tech['datetime'].dt.tz_localize(TZ, ambiguous="NaT", nonexistent="NaT")

# Eliminamos la hora 25
df_tech = df_tech[df_tech['HOUR'] <= 24]

# Identificamos las columnas relacionadas con tecnologías (todas excepto DATE, HOUR, datetime)
tech_cols = [col for col in df_tech.columns if col not in ['DATE', 'HOUR', 'datetime']]

# Renombramos las columnas para distinguirlas mejor
rename_map = {
    'COAL'                 : 'carbon',
    'COMBINED_CYCLE'       : 'ciclo_combinado',
    'FUEL_GAS'             : 'fuel_gas',
    'NUCLEAR'              : 'nuclear',
    'RESIDUALS'            : 'residuos',
    'SELF_PRODUCER'        : 'autoproduccion',
    'WIND'                 : 'eolica',
    'HYDRO'                : 'hidraulica',
    'PHOTOVOLTAIC_SOLAR'   : 'solar_fotovoltaica',
    'THERMAL_SOLAR'        : 'solar_termica',
    'IMPORT'               : 'importacion',
    'IMPORT_WITHOUT_MIBEL' : 'importacion_sin_mibel'
}

df_tech = df_tech.rename(columns=rename_map)
df_tech.columns = df_tech.columns.str.lower().str.strip()

cols_tech = [col for col in df_tech.columns if col not in ['date', 'hour', 'datetime']]

df_tech = df_tech[['datetime'] + cols_tech]
df_tech = df_tech.rename(columns={col: f"casada_{col}" for col in cols_tech})

cols_tech = [col for col in df_tech.columns if col != 'datetime']
df_tech[cols_tech] = df_tech[cols_tech].fillna(0)

df_tech = df_tech.sort_values('datetime', ignore_index=True)

## Datos en formato diario

In [21]:
# Replicamos el valor diario para cada hora (distribución proporcional entre las horas)
def day_to_hour(df, group_key):
    out = []
    for k, g in df.groupby(group_key):
        g     = g.sort_values('date')
        hours = pd.date_range(start=g['date'].min(), end=g['date'].max() + pd.Timedelta(hours=23), freq='h', tz=TZ)
        g     = g.set_index('date').reindex(hours.floor('D')).ffill()
        
        g['datetime'] = hours
        g['valor']    = g['valor'] / 24.0
        g[group_key]  = k

        out.append(g.reset_index(drop=True))
        
    return pd.concat(out, ignore_index=True)

In [22]:
print(f"Posibles valores para las tecnologías empleadas en el almacenamiento:\n{df_almacenamiento['tecnologia'].unique()}")

Posibles valores para las tecnologías empleadas en el almacenamiento:
['Consumo bombeo' 'Turbinación bombeo' 'Carga batería' 'Entrega batería']


In [23]:
# Normalizamos los nombres
map_alm = {
    'Consumo bombeo'     : 'consumo_bombeo',
    'Turbinación bombeo' : 'turbinacion_bombeo',
    'Carga batería'      : 'carga_bateria',
    'Entrega batería'    : 'entrega_bateria'
}

df_almacenamiento['tecnologia'] = df_almacenamiento['tecnologia'].map(map_alm)

# Conversión a datetime en hora peninsular
df_almacenamiento['datetime'] = pd.to_datetime(df_almacenamiento['datetime'], utc=True)
df_almacenamiento['datetime'] = df_almacenamiento['datetime'].dt.tz_convert(TZ)
df_almacenamiento['date']     = df_almacenamiento['datetime'].dt.floor('D')

# Replicamos el valor diario para cada hora (distribución proporcional entre las horas)
df_almacenamiento = day_to_hour(df_almacenamiento, 'tecnologia')

# Pivotamos a formato ancho (1 fila por hora)
df_almacenamiento = df_almacenamiento.pivot_table(index='datetime', columns='tecnologia', values='valor', aggfunc='sum').reset_index()

# Prefijo para diferenciar los datos
cols = [c for c in df_almacenamiento.columns if c != 'datetime']
df_almacenamiento = df_almacenamiento.rename(columns={c: f"alm_{c}" for c in cols})

# Relleno de nulos
cols = [c for c in df_almacenamiento.columns if c != 'datetime']
df_almacenamiento[cols] = df_almacenamiento[cols].fillna(0)

# Ordenamos por la fecha
df_almacenamiento = df_almacenamiento.sort_values('datetime', ignore_index=True)

In [24]:
print(f"Posibles valores para los países que realizan intercambios de energía con la península:\n{df_intercambio['pais'].unique()}")
print(f"Posibles valores para la denominación de intercambios de energía con la península:\n{df_intercambio['tipo'].unique()}")

Posibles valores para los países que realizan intercambios de energía con la península:
['Andorra' 'Francia' 'Marruecos' 'Portugal']
Posibles valores para la denominación de intercambios de energía con la península:
['Exportación' 'Importación']


In [25]:
# Normalizamos los nombres
df_intercambio['pais'] = df_intercambio['pais'].str.lower().str.strip()
df_intercambio['tipo'] = df_intercambio['tipo'].str.lower().str.strip()

map_tipo = {
    'exportación': 'export',
    'importación': 'import'
}

# Juntamos el tipo de intercambio con el nombre del país para las futuras columnas
df_intercambio['tipo'] = df_intercambio['tipo'].map(map_tipo)
df_intercambio['key']  = df_intercambio['pais'] + "_" + df_intercambio['tipo']

# Conversión a datetime en hora peninsular
df_intercambio['datetime'] = pd.to_datetime(df_intercambio['datetime'], utc=True)
df_intercambio['datetime'] = df_intercambio['datetime'].dt.tz_convert(TZ)
df_intercambio['date']     = df_intercambio['datetime'].dt.floor('D')

# Replicamos el valor diario para cada hora (distribución proporcional entre las horas)
df_intercambio = day_to_hour(df_intercambio, 'key')

# Pivotamos a formato ancho (1 fila por hora)
df_intercambio = df_intercambio.pivot_table(index='datetime', columns='key', values='valor', aggfunc='sum').reset_index()

# Relleno de nulos
cols = [c for c in df_intercambio.columns if c != 'datetime']
df_intercambio[cols] = df_intercambio[cols].fillna(0)

# Ordenamos por la fecha
df_intercambio = df_intercambio.sort_values('datetime', ignore_index=True)

In [26]:
print("Posibles valores para las tecnologías empleadas en la generación según tipo de energía:")
for tipo, group in df_generacion.groupby('tipo_energia'):
    tecnologias = sorted(group['tecnologia'].dropna().unique())

    print(f"Tipo de energía: {tipo} -> Tecnologías: {tecnologias}")

Posibles valores para las tecnologías empleadas en la generación según tipo de energía:
Tipo de energía: No-Renovable -> Tecnologías: ['Carbón', 'Ciclo combinado', 'Cogeneración', 'Fuel + Gas', 'Nuclear', 'Residuos no renovables', 'Turbina de vapor']
Tipo de energía: Renovable -> Tecnologías: ['Eólica', 'Hidráulica', 'Otras renovables', 'Residuos renovables', 'Solar fotovoltaica', 'Solar térmica']


In [27]:
# Normalizamos los nombres
df_generacion['tipo_energia'] = df_generacion['tipo_energia'].str.lower().str.strip()
df_generacion['tecnologia']   = df_generacion['tecnologia'].str.lower().str.strip()

map_tech = {
    # No renovables
    'carbón'                 : 'carbon',
    'ciclo combinado'        : 'ciclo_combinado',
    'cogeneración'           : 'cogeneracion',
    'fuel + gas'             : 'fuel_gas',
    'nuclear'                : 'nuclear',
    'residuos no renovables' : 'residuos_no_renovables',
    'turbina de vapor'       : 'turbina_vapor',
    # Renovables
    'eólica'                 : 'eolica',
    'hidráulica'             : 'hidraulica',
    'otras renovables'       : 'otras_renovables',
    'residuos renovables'    : 'residuos_renovables',
    'solar fotovoltaica'     : 'solar_fotovoltaica',
    'solar térmica'          : 'solar_termica'
}

df_generacion['tecnologia'] = df_generacion['tecnologia'].map(map_tech)

# Conversión a datetime en hora peninsular
df_generacion['datetime'] = pd.to_datetime(df_generacion['datetime'], utc=True)
df_generacion['datetime'] = df_generacion['datetime'].dt.tz_convert(TZ)
df_generacion['date']     = df_generacion['datetime'].dt.floor('D')

# Replicamos el valor diario para cada hora (distribución proporcional entre las horas)
df_generacion = day_to_hour(df_generacion, 'tecnologia')

# Pivotamos a formato ancho (1 fila por hora)
df_generacion = df_generacion.pivot_table(index='datetime',columns='tecnologia', values='valor', aggfunc='sum').reset_index()

# Prefijo para diferenciar los datos
cols = [c for c in df_generacion.columns if c != 'datetime']
df_generacion = df_generacion.rename(columns={c: f"gen_{c}" for c in cols})

# Relleno de nulos
cols = [c for c in df_generacion.columns if c != 'datetime']
df_generacion[cols] = df_generacion[cols].fillna(0)

# Ordenamos por la fecha
df_generacion = df_generacion.sort_values('datetime', ignore_index=True)

# Revisión tras preprocesado y correcciones

In [28]:
dfs = {
    "Demanda"                   : df_demanda,
    "Precio marginal español"   : df_price_sp,
    "Energia casada ibérica"    : df_ener_ib,
    "Energía casada por tech"   : df_tech,
    "Almacenamiento"            : df_almacenamiento,
    "Intercambios"              : df_intercambio,
    "Energía generada por tech" : df_generacion,
}

In [29]:
print("Columnas de cada dataset:")
for name, df in dfs.items():
    print(f"\n{name}:")
    print(df.columns.tolist())

Columnas de cada dataset:

Demanda:
['datetime', 'demanda']

Precio marginal español:
['datetime', 'precio']

Energia casada ibérica:
['datetime', 'energia_mercado']

Energía casada por tech:
['datetime', 'casada_carbon', 'casada_fuel_gas', 'casada_autoproduccion', 'casada_nuclear', 'casada_hidraulica', 'casada_ciclo_combinado', 'casada_eolica', 'casada_solar_termica', 'casada_solar_fotovoltaica', 'casada_residuos', 'casada_importacion', 'casada_importacion_sin_mibel']

Almacenamiento:
['datetime', 'alm_carga_bateria', 'alm_consumo_bombeo', 'alm_entrega_bateria', 'alm_turbinacion_bombeo']

Intercambios:
['datetime', 'andorra_export', 'andorra_import', 'francia_export', 'francia_import', 'marruecos_export', 'marruecos_import', 'portugal_export', 'portugal_import']

Energía generada por tech:
['datetime', 'gen_carbon', 'gen_ciclo_combinado', 'gen_cogeneracion', 'gen_eolica', 'gen_fuel_gas', 'gen_hidraulica', 'gen_nuclear', 'gen_otras_renovables', 'gen_residuos_no_renovables', 'gen_residu

In [30]:
print("Comprobación de filas duplicadas:")
for name, df in dfs.items():
    duplicados = df.duplicated().sum()

    print(f"{name:26s}: {duplicados} filas duplicadas")

Comprobación de filas duplicadas:
Demanda                   : 0 filas duplicadas
Precio marginal español   : 0 filas duplicadas
Energia casada ibérica    : 0 filas duplicadas
Energía casada por tech   : 0 filas duplicadas
Almacenamiento            : 0 filas duplicadas
Intercambios              : 0 filas duplicadas
Energía generada por tech : 0 filas duplicadas


In [31]:
print("Comprobación de horas duplicadas:")
for name, df in dfs.items():
    print(f"{name:26s}: {df['datetime'].duplicated().sum()} fechas duplicadas")

Comprobación de horas duplicadas:
Demanda                   : 0 fechas duplicadas
Precio marginal español   : 13 fechas duplicadas
Energia casada ibérica    : 12 fechas duplicadas
Energía casada por tech   : 11 fechas duplicadas
Almacenamiento            : 0 fechas duplicadas
Intercambios              : 0 fechas duplicadas
Energía generada por tech : 0 fechas duplicadas


In [32]:
print("Comprobación de valores nulos:")
for name, df in dfs.items():
    nulls = df.isnull().sum()
    cols_con_nulos = nulls[nulls > 0]

    if cols_con_nulos.empty:
        continue

    print(f"\n{name}")
    print(cols_con_nulos)

Comprobación de valores nulos:

Precio marginal español
datetime    14
precio       7
dtype: int64

Energia casada ibérica
datetime           13
energia_mercado     7
dtype: int64

Energía casada por tech
datetime    12
dtype: int64


In [33]:
for name, df in dfs.items():
    duplicated_dates = df[df['datetime'].duplicated(keep=False)]['datetime'].unique()
    if len(duplicated_dates) > 0:
        print(f"\nDataset afectado: {name}")
        print(f"Fechas encontradas: {list(duplicated_dates)}")


Dataset afectado: Precio marginal español
Fechas encontradas: [NaT]

Dataset afectado: Energia casada ibérica
Fechas encontradas: [NaT]

Dataset afectado: Energía casada por tech
Fechas encontradas: [NaT]


In [34]:
for name, df in dfs.items():

    cols_valor = [c for c in df.columns if c != 'datetime']
    mask_null  = df[cols_valor].isnull().any(axis=1)
    idx_null   = df.loc[mask_null].index

    if len(idx_null) > 0:

        print(f"\n{name}")

        rows = []
        for idx in idx_null:
            rows.extend([idx - 1, idx, idx + 1])

        rows = sorted(set(i for i in rows if i >= 0 and i < len(df)))
        display(df.loc[rows].reset_index(drop=True))


Precio marginal español


,datetime,precio
0,2019-03-31 22:00:00+02:00,56.04
1,2019-03-31 23:00:00+02:00,NaN
2,2019-04-01 00:00:00+02:00,59.00
3,2020-03-29 22:00:00+02:00,20.59
4,2020-03-29 23:00:00+02:00,NaN
5,2020-03-30 00:00:00+02:00,17.00
6,2021-03-28 22:00:00+02:00,47.07
7,2021-03-28 23:00:00+02:00,NaN
8,2021-03-29 00:00:00+02:00,39.07
9,2022-03-27 22:00:00+02:00,242.90



Energia casada ibérica


,datetime,energia_mercado
0,2019-03-31 22:00:00+02:00,23355.2
1,2019-03-31 23:00:00+02:00,NaN
2,2019-04-01 00:00:00+02:00,20592.7
3,2020-03-29 22:00:00+02:00,29736.4
4,2020-03-29 23:00:00+02:00,NaN
5,2020-03-30 00:00:00+02:00,23066.8
6,2021-03-28 22:00:00+02:00,21340.0
7,2021-03-28 23:00:00+02:00,NaN
8,2021-03-29 00:00:00+02:00,20205.6
9,2022-03-27 22:00:00+02:00,20209.7


In [35]:
# Corregimos los valores nulos del precio y la energía de mercado interpolando datos
df_price_sp['precio']         = df_price_sp['precio'].interpolate(method='linear')
df_ener_ib['energia_mercado'] = df_ener_ib['energia_mercado'].interpolate(method='linear')

In [36]:
# Eliminamos las fechas nulas en los datasets correspondientes
df_price_sp = df_price_sp.dropna(subset=['datetime']).reset_index(drop=True)
df_ener_ib  = df_ener_ib.dropna(subset=['datetime']).reset_index(drop=True)
df_tech     = df_tech.dropna(subset=['datetime']).reset_index(drop=True)

In [37]:
print("Columnas con todos sus valores a 0:")
for name, df in dfs.items():
    cols_all_zero = df.columns[(df.fillna(0) == 0).all()]

    if cols_all_zero.empty:
        continue

    print(f"\n{name}")
    print(cols_all_zero.tolist())

Columnas con todos sus valores a 0:

Energía casada por tech
['casada_fuel_gas', 'casada_autoproduccion']


In [38]:
# Eliminamos columnas sin información relevante
df_tech = df_tech.drop(columns=['casada_fuel_gas', 'casada_autoproduccion'])

In [39]:
dfs = {
    "Demanda"                   : df_demanda,
    "Precio marginal español"   : df_price_sp,
    "Energia casada ibérica"    : df_ener_ib,
    "Energía casada por tech"   : df_tech,
    "Almacenamiento"            : df_almacenamiento,
    "Intercambios"              : df_intercambio,
    "Energía generada por tech" : df_generacion,
}

In [40]:
for name, df in dfs.items():
    nulls = df.isnull().sum()
    cols_con_nulos = nulls[nulls > 0]

    if cols_con_nulos.empty:
        continue

    print(f"\n{name}")
    print(cols_con_nulos)

In [41]:
for name, df in dfs.items():
    duplicated_dates = df[df['datetime'].duplicated(keep=False)]['datetime'].unique()
    if len(duplicated_dates) > 0:
        print(f"Dataset afectado: {name}")
        print(f"Fechas encontradas: {list(duplicated_dates)}\n")

In [42]:
print("Dimensiones de cada dataset:")
for name, df in dfs.items():
    print(f"{name:25s} : {df.shape}")

Dimensiones de cada dataset:
Demanda                   : (61368, 2)
Precio marginal español   : (61354, 2)
Energia casada ibérica    : (59147, 2)
Energía casada por tech   : (53958, 11)
Almacenamiento            : (61368, 5)
Intercambios              : (61368, 9)
Energía generada por tech : (61368, 14)


In [43]:
print("Tipado de las columnas de cada dataset:")
for name, df in dfs.items():
    print(f"\n{name}:")
    print(df.dtypes)

Tipado de las columnas de cada dataset:

Demanda:
datetime    datetime64[ns, Europe/Madrid]
demanda                           float64
dtype: object

Precio marginal español:
datetime    datetime64[ns, Europe/Madrid]
precio                            float64
dtype: object

Energia casada ibérica:
datetime           datetime64[ns, Europe/Madrid]
energia_mercado                          float64
dtype: object

Energía casada por tech:
datetime                        datetime64[ns, Europe/Madrid]
casada_carbon                                         float64
casada_nuclear                                        float64
casada_hidraulica                                     float64
casada_ciclo_combinado                                float64
casada_eolica                                         float64
casada_solar_termica                                  float64
casada_solar_fotovoltaica                             float64
casada_residuos                                       float64
casada_

In [44]:
print("Comprobación de rango horario:")
for name, df in dfs.items():
    print(f"{name:26s}: {df['datetime'].min()} --> {df['datetime'].max()}")

Comprobación de rango horario:
Demanda                   : 2019-01-01 00:00:00+01:00 --> 2025-12-31 23:00:00+01:00
Precio marginal español   : 2019-01-01 00:00:00+01:00 --> 2025-12-31 23:00:00+01:00
Energia casada ibérica    : 2019-01-01 00:00:00+01:00 --> 2025-09-30 23:00:00+02:00
Energía casada por tech   : 2019-01-01 00:00:00+01:00 --> 2025-02-26 23:00:00+01:00
Almacenamiento            : 2019-01-01 00:00:00+01:00 --> 2025-12-31 23:00:00+01:00
Intercambios              : 2019-01-01 00:00:00+01:00 --> 2025-12-31 23:00:00+01:00
Energía generada por tech : 2019-01-01 00:00:00+01:00 --> 2025-12-31 23:00:00+01:00


In [45]:
print("Comprobación de diferencia temporal entre cada fila (hora):")
for name, df in dfs.items():
    print(f"\n{name}")
    print(df['datetime'].sort_values().diff().value_counts().head())

Comprobación de diferencia temporal entre cada fila (hora):

Demanda
datetime
0 days 01:00:00    61367
Name: count, dtype: int64

Precio marginal español
datetime
0 days 01:00:00    61346
0 days 03:00:00        7
Name: count, dtype: int64

Energia casada ibérica
datetime
0 days 01:00:00    59140
0 days 03:00:00        6
Name: count, dtype: int64

Energía casada por tech
datetime
0 days 01:00:00    53945
0 days 02:00:00        6
0 days 03:00:00        6
Name: count, dtype: int64

Almacenamiento
datetime
0 days 01:00:00    61367
Name: count, dtype: int64

Intercambios
datetime
0 days 01:00:00    61367
Name: count, dtype: int64

Energía generada por tech
datetime
0 days 01:00:00    61367
Name: count, dtype: int64


In [46]:
for name, df in dfs.items():

    df_sorted = df.sort_values('datetime').reset_index(drop=True)
    df_sorted['datetime_prev'] = df_sorted['datetime'].shift(1)
    df_sorted['diff'] = df_sorted['datetime'] - df_sorted['datetime_prev']

    saltos = df_sorted.loc[df_sorted['diff'] > pd.Timedelta(hours=1), ['datetime_prev', 'datetime', 'diff']]
    if len(saltos) > 0:
        print(f"\n{name}")
        display(saltos)


Precio marginal español

,datetime_prev,datetime,diff
7177,2019-10-27 01:00:00+02:00,2019-10-27 03:00:00+01:00,0 days 03:00:00
15911,2020-10-25 01:00:00+02:00,2020-10-25 03:00:00+01:00,0 days 03:00:00
24813,2021-10-31 01:00:00+02:00,2021-10-31 03:00:00+01:00,0 days 03:00:00
33547,2022-10-30 01:00:00+02:00,2022-10-30 03:00:00+01:00,0 days 03:00:00
42281,2023-10-29 01:00:00+02:00,2023-10-29 03:00:00+01:00,0 days 03:00:00
51015,2024-10-27 01:00:00+02:00,2024-10-27 03:00:00+01:00,0 days 03:00:00
59749,2025-10-26 01:00:00+02:00,2025-10-26 03:00:00+01:00,0 days 03:00:00



Energia casada ibérica


,datetime_prev,datetime,diff
7177,2019-10-27 01:00:00+02:00,2019-10-27 03:00:00+01:00,0 days 03:00:00
15911,2020-10-25 01:00:00+02:00,2020-10-25 03:00:00+01:00,0 days 03:00:00
24813,2021-10-31 01:00:00+02:00,2021-10-31 03:00:00+01:00,0 days 03:00:00
33547,2022-10-30 01:00:00+02:00,2022-10-30 03:00:00+01:00,0 days 03:00:00
42281,2023-10-29 01:00:00+02:00,2023-10-29 03:00:00+01:00,0 days 03:00:00
51015,2024-10-27 01:00:00+02:00,2024-10-27 03:00:00+01:00,0 days 03:00:00



Energía casada por tech


,datetime_prev,datetime,diff
2158,2019-03-31 22:00:00+02:00,2019-04-01 00:00:00+02:00,0 days 02:00:00
7176,2019-10-27 01:00:00+02:00,2019-10-27 03:00:00+01:00,0 days 03:00:00
10891,2020-03-29 22:00:00+02:00,2020-03-30 00:00:00+02:00,0 days 02:00:00
15909,2020-10-25 01:00:00+02:00,2020-10-25 03:00:00+01:00,0 days 03:00:00
19624,2021-03-28 22:00:00+02:00,2021-03-29 00:00:00+02:00,0 days 02:00:00
24810,2021-10-31 01:00:00+02:00,2021-10-31 03:00:00+01:00,0 days 03:00:00
28357,2022-03-27 22:00:00+02:00,2022-03-28 00:00:00+02:00,0 days 02:00:00
33543,2022-10-30 01:00:00+02:00,2022-10-30 03:00:00+01:00,0 days 03:00:00
37090,2023-03-26 22:00:00+02:00,2023-03-27 00:00:00+02:00,0 days 02:00:00
42276,2023-10-29 01:00:00+02:00,2023-10-29 03:00:00+01:00,0 days 03:00:00


In [47]:
print("Comprobación de orden temporal - ¿Están las fechas en orden ascendente?:")
for name, df in dfs.items():
    ordenado = df['datetime'].is_monotonic_increasing
    print(f"{name:26s}: {ordenado}")

Comprobación de orden temporal - ¿Están las fechas en orden ascendente?:
Demanda                   : True
Precio marginal español   : True
Energia casada ibérica    : True
Energía casada por tech   : True
Almacenamiento            : True
Intercambios              : True
Energía generada por tech : True


In [48]:
df_price_sp['precio'].describe()

count    61354.000000
mean        82.191462
std         64.356849
min        -15.000000
25%         38.100000
50%         64.225000
75%        112.835000
max        700.000000
Name: precio, dtype: float64

In [49]:
df_demanda['demanda'].describe()

count    61368.000000
mean     27203.185580
std       4390.419653
min        522.770000
25%      23632.220750
50%      27229.505000
75%      30487.717250
max      41483.001000
Name: demanda, dtype: float64

In [50]:
df_generacion.drop(columns='datetime').describe()

tecnologia,gen_carbon,gen_ciclo_combinado,gen_cogeneracion,gen_eolica,gen_fuel_gas,gen_hidraulica,gen_nuclear,gen_otras_renovables,gen_residuos_no_renovables,gen_residuos_renovables,gen_solar_fotovoltaica,gen_solar_termica,gen_turbina_vapor
count,61368.000000,61368.000000,61368.000000,61368.000000,61368.000000,61368.000000,61368.000000,61368.000000,61368.000000,61368.000000,61368.000000,61368.000000,61368.000000
mean,593.373621,4814.530446,2435.163383,6586.423419,0.000007,3214.395328,6199.322886,465.842316,181.625542,77.640837,3279.318162,505.758862,18.472728
std,632.243237,2783.045363,758.730924,3522.068221,0.000036,1567.810892,991.098281,77.445753,59.171706,19.001889,2118.535167,389.478404,70.237068
min,-51.460375,713.569167,718.223708,687.061292,-0.000042,531.184667,6.330375,104.580833,42.115104,24.494938,163.679042,0.001833,0.000000
25%,284.021833,2692.139167,1831.523958,3877.727250,-0.000042,2020.871125,5621.045708,417.925875,136.580833,65.791562,1442.451500,141.992292,0.000000
50%,432.790125,4112.060292,2471.238167,5820.923167,0.000000,2869.792125,6695.486000,460.048958,188.155417,84.219062,2778.624458,434.984125,0.000000
75%,721.442958,6433.390958,3115.328708,8648.612792,0.000042,4163.190542,6979.109583,521.129833,230.613896,91.836562,4668.206500,864.161542,0.000000
max,6123.023375,15379.690792,3829.508792,18327.099542,0.000042,8147.816875,7139.537917,644.204000,293.418521,108.515250,9834.724500,1326.223542,407.221333


# Generación del dataset final

In [51]:
df_master = (
    df_demanda
    .merge(df_price_sp, on='datetime', how='left')
    .merge(df_ener_ib, on='datetime', how='left')
    .merge(df_generacion, on='datetime', how='left')
    .merge(df_tech, on='datetime', how='left')
    .merge(df_almacenamiento, on='datetime', how='left')
    .merge(df_intercambio, on='datetime', how='left')
)

In [52]:
df_master.shape

(61368, 39)

In [53]:
df_master.columns

Index(['datetime', 'demanda', 'precio', 'energia_mercado', 'gen_carbon',
       'gen_ciclo_combinado', 'gen_cogeneracion', 'gen_eolica', 'gen_fuel_gas',
       'gen_hidraulica', 'gen_nuclear', 'gen_otras_renovables',
       'gen_residuos_no_renovables', 'gen_residuos_renovables',
       'gen_solar_fotovoltaica', 'gen_solar_termica', 'gen_turbina_vapor',
       'casada_carbon', 'casada_nuclear', 'casada_hidraulica',
       'casada_ciclo_combinado', 'casada_eolica', 'casada_solar_termica',
       'casada_solar_fotovoltaica', 'casada_residuos', 'casada_importacion',
       'casada_importacion_sin_mibel', 'alm_carga_bateria',
       'alm_consumo_bombeo', 'alm_entrega_bateria', 'alm_turbinacion_bombeo',
       'andorra_export', 'andorra_import', 'francia_export', 'francia_import',
       'marruecos_export', 'marruecos_import', 'portugal_export',
       'portugal_import'],
      dtype='object')

In [54]:
df_master.dtypes

datetime                        datetime64[ns, Europe/Madrid]
demanda                                               float64
precio                                                float64
energia_mercado                                       float64
gen_carbon                                            float64
gen_ciclo_combinado                                   float64
gen_cogeneracion                                      float64
gen_eolica                                            float64
gen_fuel_gas                                          float64
gen_hidraulica                                        float64
gen_nuclear                                           float64
gen_otras_renovables                                  float64
gen_residuos_no_renovables                            float64
gen_residuos_renovables                               float64
gen_solar_fotovoltaica                                float64
gen_solar_termica                                     float64
gen_turb

In [55]:
df_master.isna().sum()

datetime                           0
demanda                            0
precio                            14
energia_mercado                 2221
gen_carbon                         0
gen_ciclo_combinado                0
gen_cogeneracion                   0
gen_eolica                         0
gen_fuel_gas                       0
gen_hidraulica                     0
gen_nuclear                        0
gen_otras_renovables               0
gen_residuos_no_renovables         0
gen_residuos_renovables            0
gen_solar_fotovoltaica             0
gen_solar_termica                  0
gen_turbina_vapor                  0
casada_carbon                   7410
casada_nuclear                  7410
casada_hidraulica               7410
casada_ciclo_combinado          7410
casada_eolica                   7410
casada_solar_termica            7410
casada_solar_fotovoltaica       7410
casada_residuos                 7410
casada_importacion              7410
casada_importacion_sin_mibel    7410
a

In [56]:
df_master['precio'] = df_master['precio'].interpolate(method='linear')

In [57]:
df_master.isna().sum()

datetime                           0
demanda                            0
precio                             0
energia_mercado                 2221
gen_carbon                         0
gen_ciclo_combinado                0
gen_cogeneracion                   0
gen_eolica                         0
gen_fuel_gas                       0
gen_hidraulica                     0
gen_nuclear                        0
gen_otras_renovables               0
gen_residuos_no_renovables         0
gen_residuos_renovables            0
gen_solar_fotovoltaica             0
gen_solar_termica                  0
gen_turbina_vapor                  0
casada_carbon                   7410
casada_nuclear                  7410
casada_hidraulica               7410
casada_ciclo_combinado          7410
casada_eolica                   7410
casada_solar_termica            7410
casada_solar_fotovoltaica       7410
casada_residuos                 7410
casada_importacion              7410
casada_importacion_sin_mibel    7410
a

In [58]:
df_master.sample(3)

,datetime,demanda,precio,energia_mercado,gen_carbon,gen_ciclo_combinado,gen_cogeneracion,gen_eolica,gen_fuel_gas,gen_hidraulica,...,alm_entrega_bateria,alm_turbinacion_bombeo,andorra_export,andorra_import,francia_export,francia_import,marruecos_export,marruecos_import,portugal_export,portugal_import
22502,2021-07-26 15:00:00+02:00,32341.125,105.00,32044.2,431.006000,5660.208125,3059.730833,4019.947375,-0.000042,2333.053583,...,0.0,75.036667,-3.721667,0.002917,-20.736958,2103.939042,-57.691667,47.541667,-580.301625,496.748125
33657,2022-11-03 09:00:00+01:00,27729.054,120.46,24384.3,538.083292,5027.057458,2061.575792,9634.555417,0.000042,1429.486333,...,0.0,590.308958,0.000000,0.000000,-576.332083,1564.289458,-167.104167,8.950000,-1117.547958,345.404625
34436,2022-12-05 20:00:00+01:00,32185.524,245.58,27193.5,1143.522917,9468.833333,1907.146917,5376.455208,0.000042,3499.819875,...,0.0,450.260458,-60.940417,0.000000,-2362.958583,89.021292,-206.075000,125.979167,-490.025750,930.080125


In [59]:
current = Path.cwd()
path = current.parent / "data" / "preprocessed_data"
path.mkdir(parents=True, exist_ok=True)

file_path = path / "preprocess_data.parquet"
df_master.to_parquet(file_path)